# Modelo multirrotor da validação do artigo

Este notebook reconstrói o modelo de `validation_paper_01.py` em etapas. A velocidade padrão é 4500 rpm e pode ser alterada na célula de parâmetros. A integração usa `backlash_ross.py`, que mantém somente as rigidezes estruturais dos dois rotores na matriz global e calcula a interação da malha pela força não linear de backlash.

## 1. Importações e diretórios

Importa o ROSS instalado no ambiente `ross3`, a variante do backlash e define o diretório dos CSVs do artigo. Não é necessário alterar `sys.path`.

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import ross as rs

from backlash_ross import Backlash

CSV_DIR = Path(r"C:\Users\M\Desktop\doutorado\teste_backlash_ross\mestrado\cvs_paper")
CSV_DIR

## 2. Parâmetros da engrenagem e da simulação

Define geometria, inércia, mancais, folga, erro de transmissão, rigidez da malha, torques e duração conforme a função `run_simulation_at_speed` do arquivo de validação.

In [ ]:
speed_rpm = 4500.0
sim_time_seconds = 2.65

z1 = z2 = 20
m_n = 0.01
pd_gear = m_n * z1
alpha_0_rad = np.radians(20.0)
width = 0.030

b0 = 50e-6
error_amp = 20e-6
m_gear = 6.57
J_gear = 0.0365

k_brg = 1.0e8
c_brg = 512.64
ks = 3.6228e8
kd = 6.5072e8

T10, T1a = 300.0, 100.0
T20, T2a = 300.0, 100.0

speed_rad_s = speed_rpm * np.pi / 30.0
shaft_period = 2.0 * np.pi / speed_rad_s
n_cicles = int(np.ceil(sim_time_seconds / shaft_period))
cut_cicles = 0

print(f"Velocidade: {speed_rpm:.0f} rpm ({speed_rad_s:.6f} rad/s)")
print(f"Revoluções simuladas: {n_cicles}")

## 3. Materiais

Cria o aço da engrenagem e o material quase rígido usado no pequeno elemento de eixo do modelo de validação.

In [ ]:
steel = rs.Material(
    name="Steel", rho=7850, E=2e11, Poisson=0.3
)
steel_stiff = rs.Material(
    name="Steel_Stiff", rho=0.01, E=1e15, Poisson=0.3
)

## 4. Eixo, mancal e engrenagem motora

Monta os elementos do primeiro rotor. Massa e inércias da engrenagem são ajustadas para os valores do artigo após a criação de `GearElementTVMS`.

In [ ]:
shaft1 = [
    rs.ShaftElement(
        L=0.0001, idl=0.0, odl=0.0001, material=steel_stiff, n=0
    )
]

bearing1 = rs.BearingElement(
    n=0, kxx=k_brg, kyy=k_brg, cxx=c_brg, cyy=c_brg
)

bore_diameter = np.sqrt(
    pd_gear**2 - (4.0 * m_gear) / (np.pi * width * steel.rho)
)

gear1 = rs.GearElementTVMS(
    n=0,
    material=steel,
    width=width,
    bore_diameter=bore_diameter,
    module=m_n,
    n_teeth=z1,
    pr_angle=alpha_0_rad,
    helix_angle=0.0,
    addendum_coeff=1.0,
    tip_clearance_coeff=0.25,
)
gear1.m = m_gear
gear1.Ip = J_gear
gear1.Id = 0.0001 * J_gear / 2.0

## 5. Rotores motor e movido

Constrói o rotor motor e cria uma cópia independente para representar o rotor movido, como no código de validação.

In [ ]:
rotor1 = rs.Rotor(
    shaft_elements=shaft1,
    disk_elements=[gear1],
    bearing_elements=[bearing1],
)
rotor2 = deepcopy(rotor1)

rotor1, rotor2

## 6. Montagem do multirrotor

Acopla os rotores pelos nós das engrenagens. Neste estágio o ROSS calcula geometria, relação de transmissão e a tabela nominal da malha. A remoção da matriz linear de acoplamento será feita na cópia interna criada pela classe `Backlash`.

In [ ]:
multirotor = rs.MultiRotor(
    driving_rotor=rotor1,
    driven_rotor=rotor2,
    coupled_nodes=(0, 0),
    update_mesh_stiffness=True,
    square_varying_stiffness={
        "enable": True,
        "amplitude_ratio": 0.275,
    },
    orientation_angle=0.0,
    position="above",
)

gear_nodes = [
    int(element.n)
    for element in multirotor.disk_elements
    if isinstance(element, rs.GearElement)
]
gear_nodes

## 7. Instância do backlash sem matriz linear de acoplamento

Cria a cópia interna do multirrotor e substitui `add_coupling_stiffness` por uma função identidade. Assim, `MultiRotor.K` conserva as matrizes dos dois rotores nas posições globais corretas, mas não soma `K_coupling * mesh.stiffness`.

In [ ]:
backlash = Backlash(
    multirotor=multirotor,
    speed_driving_gear=speed_rad_s,
    b0=b0,
    error_amp=error_amp,
    gear_mesh_stiffness=None,
    num_points_cicle=6000,
    n_cicles=n_cicles,
    cut_cicles=cut_cicles,
    use_multirotor_coupling_stiffness=False,
    compute_contact_ratio=True,
    mesh_damping_ratio=0.07,
)

## 8. Verificação da matriz de rigidez global

Compara a matriz retornada por `MultiRotor.K` com a montagem direta das matrizes dos rotores. A asserção confirma que nenhuma parcela da matriz de acoplamento foi adicionada.

In [ ]:
model = backlash.multirotor
K_global = model.K(speed_rad_s)
K_rotors = model._join_matrices(
    model.rotors["driving"].K(speed_rad_s, speed_rad_s),
    model.rotors["driven"].K(
        speed_rad_s, speed_rad_s * model.mesh.gear_ratio
    ),
)

np.testing.assert_allclose(K_global, K_rotors, rtol=0.0, atol=0.0)
print("Verificação concluída: K_global contém somente as matrizes dos rotores.")

## 9. Tabela da rigidez variável da malha

Gera a tabela quadrada com os níveis de contato duplo (`kd`) e simples (`ks`). Essa rigidez permanece disponível para o cálculo da força não linear, apesar de não ser adicionada à matriz linear global.

In [ ]:
theta_arr, cr_arr, K_table = backlash._get_or_create_stiffness_table(
    force_recalculate=True,
    square_varying_stiffness=True,
    kd=kd,
    ks=ks,
    n_poits=1000,
)

theta_arr.shape, cr_arr.shape, K_table.shape

## 10. Forças externas e torques

Cria o vetor de forças no tempo. Os torques médio e harmônico são aplicados ao grau de liberdade torsional das duas engrenagens.

In [ ]:
unb_node = gear_nodes
unb_magnitude = [0.0, 0.0]
unb_phase = [0.0, 0.0]

w1 = speed_rad_s
w2 = multirotor.mesh.gear_ratio * w1
F = np.zeros((len(backlash.time), multirotor.ndof))

torsional_dof_1 = unb_node[0] * multirotor.number_dof + 5
torsional_dof_2 = unb_node[1] * multirotor.number_dof + 5
F[:, torsional_dof_1] = T10 + T1a * np.sin(w1 * backlash.time)
F[:, torsional_dof_2] = T20 + T2a * np.sin(w2 * backlash.time)

F.shape

## 11. Integração dinâmica com backlash

Executa o Newmark interno com os parâmetros do arquivo de validação. Esta é a célula computacionalmente mais cara do notebook.

In [ ]:
gamma = 0.5
beta = 0.25 * (gamma + 0.5) ** 2

results = backlash.run_dynamic_backlash(
    unb_node=unb_node,
    unb_magnitude=unb_magnitude,
    unb_phase=unb_phase,
    integration_method="internal_newmark",
    gamma=gamma,
    beta=beta,
    tol=1e-6,
    sigma=1e5,
    smooth_operator=False,
    add_force=F,
)

results

## 12. Acesso inicial aos resultados

Mostra como acessar o deslocamento horizontal da engrenagem motora e as grandezas internas registradas pelo modelo de backlash.

In [ ]:
idx_x1 = unb_node[0] * multirotor.number_dof
x1 = results.yout[:, idx_x1]
delta = backlash.backlash_results["delta"]
mesh_force = backlash.backlash_results["Fm"]

print(f"Amostras: {len(results.t)}")
print(f"Máximo |x1|: {np.max(np.abs(x1)):.6e} m")
print(f"Máximo |Fm|: {np.max(np.abs(mesh_force)):.6e} N")